# 🫀 Kidney Tumor Identification System
### End-to-End CNN Pipeline — VGG16 + MLflow/DagsHub + MongoDB
---
**Pipeline Stages:**
1. Install dependencies & set up env secrets  
2. Build project structure (configs, YAML, src package)  
3. Data Ingestion (Google Drive → local)  
4. Prepare Base Model (VGG16 fine-tuned)  
5. Prepare Callbacks (TensorBoard + ModelCheckpoint)  
6. Training (GPU-accelerated)  
7. Evaluation + MLflow logging  
8. Metadata & scores snapshot  
9. Zip & download all outputs  

> ⚡ **Runtime:** Set to **GPU** (Runtime → Change runtime type → T4 GPU)


In [1]:
# ── : Verify GPU is available ──────────────────────────────────────────
# Colab must be set to GPU runtime: Runtime → Change runtime type → T4 GPU
import subprocess, sys

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode == 0:
    print("✅ GPU detected:")
    # Print the first few relevant lines from nvidia-smi
    for line in result.stdout.split("\n")[:12]:
        print(line)
else:
    print("⚠️  No GPU found. Go to Runtime → Change runtime type → GPU.")
    print("   Training will be slow on CPU.")

import tensorflow as tf
print(f"\n🔢 TensorFlow version : {tf.__version__}")
print(f"🖥️  GPUs available      : {tf.config.list_physical_devices('GPU')}")


✅ GPU detected:
Fri Apr 10 04:09:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------------------------

In [2]:
# ──: Install all required Python packages ──────────────────────────────
# tensorflow 2.12 is pinned for VGG16 compatibility; other versions may break
# python-box 6.0.2 is pinned for ConfigBox API used in config manager
# ensure 1.0.2 is pinned for type annotation enforcement
# gdown is needed to pull the dataset from Google Drive

import subprocess, sys

packages = [
    "tensorflow",
    "mlflow",
    "python-box",
    "ensure",
    "gdown",
    "PyYAML",
    "joblib",
    "pymongo",          # MongoDB driver for experiment metadata storage
    "dnspython",        # Required for MongoDB+srv:// connection strings
]

for pkg in packages:
    print(f"📦 Installing {pkg} ...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        stdout=subprocess.DEVNULL
    )

print("\n✅ All packages installed successfully.")


📦 Installing tensorflow ...
📦 Installing mlflow ...
📦 Installing python-box ...
📦 Installing ensure ...
📦 Installing gdown ...
📦 Installing PyYAML ...
📦 Installing joblib ...
📦 Installing pymongo ...
📦 Installing dnspython ...

✅ All packages installed successfully.


In [3]:
# ── : Load all secrets from Colab Secrets & set environment variables ──
# Secrets are stored in Colab's key-value secret store (🔑 icon in sidebar).
# Required secrets:
#   MONGO_DB_URL           – MongoDB Atlas connection string
#   MLFLOW_TRACKING_URI    – DagsHub MLflow tracking URI
#   MLFLOW_TRACKING_USERNAME – DagsHub username
#   MLFLOW_TRACKING_PASSWORD – DagsHub access token

import os
from google.colab import userdata

# ── MongoDB (from Colab Secrets) ──────────────────────────────────────────────
os.environ['MONGO_DB_URL'] = userdata.get('MONGO_DB_URL')

# ── MLflow / DagsHub ──────────────────────────────────────────────────────────
USE_DAGSHUB = True  # ✅ Set False to fall back to local file-based MLflow

if USE_DAGSHUB:
    os.environ['MLFLOW_TRACKING_URI']      = userdata.get('MLFLOW_TRACKING_URI')
    os.environ['MLFLOW_TRACKING_USERNAME'] = userdata.get('MLFLOW_TRACKING_USERNAME')
    os.environ['MLFLOW_TRACKING_PASSWORD'] = userdata.get('MLFLOW_TRACKING_PASSWORD')
else:
    # Local MLflow — experiment data saved inside the Colab VM (ephemeral)
    os.environ['MLFLOW_TRACKING_URI'] = f"file://{os.getcwd()}/mlruns"

print('✅ Env vars set.')
print(f"   MLFLOW_TRACKING_URI = {os.environ['MLFLOW_TRACKING_URI']}")


✅ Env vars set.
   MLFLOW_TRACKING_URI = https://dagshub.com/prithusarkar90/networksecurity.mlflow


In [4]:
# ── Set working directory and write all config YAML / params files ───
# All artifacts will be saved relative to PROJECT_ROOT inside /content/
# config.yaml holds all path definitions; params.yaml holds hyperparameters.

import os
from pathlib import Path

# ── Working directory ─────────────────────────────────────────────────────────
PROJECT_ROOT = "/content/KidneyTumorSystem"
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.chdir(PROJECT_ROOT)
print(f"📁 Working directory set to: {os.getcwd()}")

# ── config/config.yaml ────────────────────────────────────────────────────────
os.makedirs("config", exist_ok=True)
config_yaml = """\
artifacts_root: artifacts

data_ingestion:
  root_dir: artifacts/data_ingestion
  source_URL: https://drive.google.com/file/d/1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3/view?usp=sharing
  local_data_file: artifacts/data_ingestion/data.zip
  unzip_dir: artifacts/data_ingestion

prepare_base_model:
  root_dir: artifacts/prepare_base_model
  base_model_path: artifacts/prepare_base_model/base_model.h5
  updated_base_model_path: artifacts/prepare_base_model/base_model_updated.h5

prepare_callbacks:
  root_dir: artifacts/prepare_callbacks
  tensorboard_root_log_dir: artifacts/prepare_callbacks/tensorboard_log_dir
  checkpoint_model_filepath: artifacts/prepare_callbacks/checkpoint_dir/model.h5

training:
  root_dir: artifacts/training
  trained_model_path: artifacts/training/model.h5
"""

with open("config/config.yaml", "w") as f:
    f.write(config_yaml)

# ── params.yaml ───────────────────────────────────────────────────────────────
params_yaml = """\
AUGMENTATION: True
IMAGE_SIZE: [224, 224, 3]   # VGG16 expected input
BATCH_SIZE: 16
INCLUDE_TOP: False           # Exclude VGG16's final classification layer
EPOCHS: 1                    # Increase for production runs
CLASSES: 2                   # Tumor vs Normal
WEIGHTS: imagenet            # Transfer-learning from ImageNet
LEARNING_RATE: 0.01
"""

with open("params.yaml", "w") as f:
    f.write(params_yaml)

print("✅ config/config.yaml written")
print("✅ params.yaml written")


📁 Working directory set to: /content/KidneyTumorSystem
✅ config/config.yaml written
✅ params.yaml written


In [5]:
# ── CELL 5: Write the cnnClassifier Python package source files ───────────────
# Instead of cloning a repo, we write every module inline so the notebook is
# fully self-contained and can run offline (except for data download & MLflow).

import os
from pathlib import Path

# ── Directory skeleton ────────────────────────────────────────────────────────
pkg_dirs = [
    "src/cnnClassifier",
    "src/cnnClassifier/components",
    "src/cnnClassifier/config",
    "src/cnnClassifier/constants",
    "src/cnnClassifier/entity",
    "src/cnnClassifier/pipeline",
    "src/cnnClassifier/utils",
    "logs",
    "artifacts",
]
for d in pkg_dirs:
    os.makedirs(d, exist_ok=True)

# Helper: write a file and report
def write_file(path, content):
    Path(path).write_text(content)
    print(f"  ✅ {path}")

print("Writing package source files...")

# ── src/cnnClassifier/__init__.py  (logger setup) ────────────────────────────
write_file("src/cnnClassifier/__init__.py", """\
import os, sys, logging

logging_str = "[%(asctime)s: %(levelname)s: %(module)s: %(message)s]"
log_dir = "logs"
log_filepath = os.path.join(log_dir, "running_logs.log")
os.makedirs(log_dir, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format=logging_str,
    handlers=[
        logging.FileHandler(log_filepath),
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger("cnnClassifierLogger")
""")

# ── sub-package __init__.py stubs ─────────────────────────────────────────────
for stub in [
    "src/cnnClassifier/components/__init__.py",
    "src/cnnClassifier/config/__init__.py",
    "src/cnnClassifier/constants/__init__.py",
    "src/cnnClassifier/entity/__init__.py",
    "src/cnnClassifier/pipeline/__init__.py",
    "src/cnnClassifier/utils/__init__.py",
]:
    write_file(stub, "")

# ── constants ─────────────────────────────────────────────────────────────────
write_file("src/cnnClassifier/constants/__init__.py", """\
from pathlib import Path

CONFIG_FILE_PATH = Path("config/config.yaml")
PARAMS_FILE_PATH = Path("params.yaml")
""")

# ── utils/common.py ───────────────────────────────────────────────────────────
write_file("src/cnnClassifier/utils/common.py", """\
import os, json, joblib, base64
from box.exceptions import BoxValueError
import yaml
from cnnClassifier import logger
from ensure import ensure_annotations
from box import ConfigBox
from pathlib import Path
from typing import Any

@ensure_annotations
def read_yaml(path_to_yaml: Path) -> ConfigBox:
    \'\'\'Read YAML file and return as ConfigBox (dot-accessible dict).\'\'\'
    try:
        with open(path_to_yaml) as f:
            content = yaml.safe_load(f)
            logger.info(f"yaml file: {path_to_yaml} loaded successfully")
            return ConfigBox(content)
    except BoxValueError:
        raise ValueError("yaml file is empty")
    except Exception as e:
        raise e

@ensure_annotations
def create_directories(path_to_directories: list, verbose=True):
    \'\'\'Create a list of directories (mkdir -p equivalent).\'\'\'
    for path in path_to_directories:
        os.makedirs(path, exist_ok=True)
        if verbose:
            logger.info(f"created directory at: {path}")

@ensure_annotations
def save_json(path: Path, data: dict):
    \'\'\'Serialize a dict to a JSON file with 4-space indentation.\'\'\'
    with open(path, "w") as f:
        json.dump(data, f, indent=4)
    logger.info(f"json file saved at: {path}")

@ensure_annotations
def load_json(path: Path) -> ConfigBox:
    \'\'\'Load a JSON file and return as ConfigBox.\'\'\'
    with open(path) as f:
        content = json.load(f)
    logger.info(f"json file loaded from: {path}")
    return ConfigBox(content)

@ensure_annotations
def save_bin(data: Any, path: Path):
    \'\'\'Persist arbitrary Python object as a joblib binary file.\'\'\'
    joblib.dump(value=data, filename=path)
    logger.info(f"binary file saved at: {path}")

@ensure_annotations
def load_bin(path: Path) -> Any:
    \'\'\'Load a joblib binary file.\'\'\'
    data = joblib.load(path)
    logger.info(f"binary file loaded from: {path}")
    return data

@ensure_annotations
def get_size(path: Path) -> str:
    \'\'\'Return file size in KB as a human-readable string.\'\'\'
    size_in_kb = round(os.path.getsize(path) / 1024)
    return f"~ {size_in_kb} KB"

def decodeImage(imgstring, fileName):
    \'\'\'Decode a base64 image string and write it to disk.\'\'\'
    imgdata = base64.b64decode(imgstring)
    with open(fileName, "wb") as f:
        f.write(imgdata)

def encodeImageIntoBase64(croppedImagePath):
    \'\'\'Read an image file and return its base64-encoded bytes.\'\'\'
    with open(croppedImagePath, "rb") as f:
        return base64.b64encode(f.read())
""")

# ── entity/config_entity.py ───────────────────────────────────────────────────
write_file("src/cnnClassifier/entity/config_entity.py", """\
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    \'\'\'Paths and URL for the data download + extraction step.\'\'\'
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

@dataclass(frozen=True)
class PrepareBaseModelConfig:
    \'\'\'VGG16 base-model build settings (architecture + hyperparams).\'\'\'
    root_dir: Path
    base_model_path: Path
    updated_base_model_path: Path
    params_image_size: list
    params_learning_rate: float
    params_include_top: bool
    params_weights: str
    params_classes: int

@dataclass(frozen=True)
class PrepareCallbacksConfig:
    \'\'\'Paths used to persist TensorBoard logs and model checkpoints.\'\'\'
    root_dir: Path
    tensorboard_root_log_dir: Path
    checkpoint_model_filepath: Path

@dataclass(frozen=True)
class TrainingConfig:
    \'\'\'All paths and hyperparameters required to run model training.\'\'\'
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list

@dataclass(frozen=True)
class EvaluationConfig:
    \'\'\'Paths and URIs needed for model evaluation and MLflow logging.\'\'\'
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int
""")

# ── config/configuration.py ───────────────────────────────────────────────────
write_file("src/cnnClassifier/config/configuration.py", """\
import os
from pathlib import Path
from cnnClassifier.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from cnnClassifier.utils.common import read_yaml, create_directories
from cnnClassifier.entity.config_entity import (
    DataIngestionConfig, PrepareBaseModelConfig,
    PrepareCallbacksConfig, TrainingConfig, EvaluationConfig
)

class ConfigurationManager:
    \'\'\'Central manager: reads config.yaml + params.yaml and produces
    typed config dataclasses for each pipeline stage.\'\'\'

    def __init__(self,
                 config_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])
        return DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

    def get_prepare_base_model_config(self) -> PrepareBaseModelConfig:
        config = self.config.prepare_base_model
        create_directories([config.root_dir])
        return PrepareBaseModelConfig(
            root_dir=Path(config.root_dir),
            base_model_path=Path(config.base_model_path),
            updated_base_model_path=Path(config.updated_base_model_path),
            params_image_size=self.params.IMAGE_SIZE,
            params_learning_rate=self.params.LEARNING_RATE,
            params_include_top=self.params.INCLUDE_TOP,
            params_weights=self.params.WEIGHTS,
            params_classes=self.params.CLASSES
        )

    def get_prepare_callback_config(self) -> PrepareCallbacksConfig:
        config = self.config.prepare_callbacks
        ckpt_dir = os.path.dirname(config.checkpoint_model_filepath)
        create_directories([Path(ckpt_dir), Path(config.tensorboard_root_log_dir)])
        return PrepareCallbacksConfig(
            root_dir=Path(config.root_dir),
            tensorboard_root_log_dir=Path(config.tensorboard_root_log_dir),
            checkpoint_model_filepath=Path(config.checkpoint_model_filepath)
        )

    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        # Dataset lives under the unzip dir after extraction
        training_data = os.path.join(
            self.config.data_ingestion.unzip_dir, "kidney-ct-scan-image"
        )
        create_directories([Path(training.root_dir)])
        return TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            training_data=Path(training_data),
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE
        )

    def get_evaluation_config(self) -> EvaluationConfig:
        # MLflow URI is read from env so it respects the USE_DAGSHUB toggle
        import os as _os
        return EvaluationConfig(
            path_of_model="artifacts/training/model.h5",
            training_data="artifacts/data_ingestion/kidney-ct-scan-image",
            mlflow_uri=_os.environ.get("MLFLOW_TRACKING_URI", ""),
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
""")

# ── components/data_ingestion.py ──────────────────────────────────────────────
write_file("src/cnnClassifier/components/data_ingestion.py", """\
import os, zipfile
import gdown
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size
from cnnClassifier.entity.config_entity import DataIngestionConfig
from pathlib import Path

class DataIngestion:
    \'\'\'Downloads the kidney CT-scan dataset from Google Drive and extracts it.\'\'\'

    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self) -> None:
        \'\'\'Pull the zip archive from Google Drive using gdown.\'\'\'
        try:
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file
            os.makedirs("artifacts/data_ingestion", exist_ok=True)
            logger.info(f"Downloading data from {dataset_url}")

            # Extract the file ID from the standard Google Drive share URL
            file_id = dataset_url.split("/")[-2]
            prefix = "https://drive.google.com/uc?/export=download&id="
            gdown.download(prefix + file_id, str(zip_download_dir))

            logger.info(f"Download complete → {zip_download_dir}")
        except Exception as e:
            raise e

    def extract_zip_file(self) -> None:
        \'\'\'Extract the downloaded zip into the configured unzip directory.\'\'\'
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, "r") as zip_ref:
            zip_ref.extractall(unzip_path)
        logger.info(f"Extraction complete → {unzip_path}")
""")

# ── components/prepare_base_model.py ─────────────────────────────────────────
write_file("src/cnnClassifier/components/prepare_base_model.py", """\
import tensorflow as tf
from pathlib import Path
from cnnClassifier.entity.config_entity import PrepareBaseModelConfig

class PrepareBaseModel:
    \'\'\'Load VGG16 (ImageNet weights, no top) and attach a custom classification head.\'\'\'

    def __init__(self, config: PrepareBaseModelConfig):
        self.config = config

    def get_base_model(self):
        \'\'\'Download VGG16 pre-trained weights and save the base (no head) model.\'\'\'
        self.model = tf.keras.applications.vgg16.VGG16(
            input_shape=self.config.params_image_size,
            weights=self.config.params_weights,   # "imagenet"
            include_top=self.config.params_include_top  # False → strip FC layers
        )
        self.save_model(path=self.config.base_model_path, model=self.model)

    @staticmethod
    def _prepare_full_model(model, classes, freeze_all, freeze_till, learning_rate):
        \'\'\'Freeze convolutional layers and attach a 2-class softmax head.\'\'\'
        if freeze_all:
            for layer in model.layers:
                model.trainable = False
        elif freeze_till is not None and freeze_till > 0:
            for layer in model.layers[:-freeze_till]:
                model.trainable = False

        flatten_in = tf.keras.layers.Flatten()(model.output)
        prediction = tf.keras.layers.Dense(
            units=classes,
            activation="softmax"
        )(flatten_in)

        full_model = tf.keras.models.Model(
            inputs=model.input,
            outputs=prediction
        )
        full_model.compile(
            optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"]
        )
        full_model.summary()
        return full_model

    def update_base_model(self):
        \'\'\'Apply the custom head and save the updated (trainable) model.\'\'\'
        self.full_model = self._prepare_full_model(
            model=self.model,
            classes=self.config.params_classes,
            freeze_all=True,
            freeze_till=None,
            learning_rate=self.config.params_learning_rate
        )
        self.save_model(path=self.config.updated_base_model_path, model=self.full_model)

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)
""")

# ── components/prepare_callbacks.py ──────────────────────────────────────────
write_file("src/cnnClassifier/components/prepare_callbacks.py", """\
import os, time
import tensorflow as tf
from cnnClassifier.entity.config_entity import PrepareCallbacksConfig

class PrepareCallback:
    \'\'\'Factory for Keras training callbacks: TensorBoard + ModelCheckpoint.\'\'\'

    def __init__(self, config: PrepareCallbacksConfig):
        self.config = config

    @property
    def _create_tb_callbacks(self):
        \'\'\'TensorBoard callback — writes logs to a timestamped sub-directory.\'\'\'
        timestamp = time.strftime("%Y-%m-%d-%H-%M-%S")
        tb_running_log_dir = os.path.join(
            self.config.tensorboard_root_log_dir,
            f"tb_logs_at_{timestamp}"
        )
        return tf.keras.callbacks.TensorBoard(log_dir=tb_running_log_dir)

    @property
    def _create_ckpt_callbacks(self):
        \'\'\'ModelCheckpoint callback — saves best weights only (val_loss).\'\'\'
        return tf.keras.callbacks.ModelCheckpoint(
            filepath=str(self.config.checkpoint_model_filepath),
            save_best_only=True
        )

    def get_tb_ckpt_callbacks(self):
        \'\'\'Return both callbacks as a list ready to pass to model.fit().\'\'\'
        return [self._create_tb_callbacks, self._create_ckpt_callbacks]
""")

# ── components/training.py ────────────────────────────────────────────────────
write_file("src/cnnClassifier/components/training.py", """\
import tensorflow as tf
from pathlib import Path
from cnnClassifier.entity.config_entity import TrainingConfig

class Training:
    \'\'\'Trains the updated VGG16 model on kidney CT-scan images.\'\'\'

    def __init__(self, config: TrainingConfig):
        self.config = config

    def get_base_model(self):
        \'\'\'Load the updated (head-attached) base model from disk.\'\'\'
        self.model = tf.keras.models.load_model(self.config.updated_base_model_path)

    def train_valid_generator(self):
        \'\'\'Build ImageDataGenerators for train/val splits.
        Augmentation is applied to training data only when enabled in params.\'\'\'
        datagenerator_kwargs = dict(rescale=1.0 / 255, validation_split=0.20)
        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],  # (224, 224)
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        # Validation generator — no augmentation
        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )
        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        # Training generator — augmented if flag is set
        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

    def train(self, callback_list: list):
        \'\'\'Run model.fit() for the configured number of epochs with callbacks.\'\'\'
        self.steps_per_epoch = (
            self.train_generator.samples // self.train_generator.batch_size
        )
        self.validation_steps = (
            self.valid_generator.samples // self.valid_generator.batch_size
        )

        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps=self.validation_steps,
            validation_data=self.valid_generator,
            callbacks=callback_list
        )
        self.save_model(path=self.config.trained_model_path, model=self.model)
""")

# ── components/evaluation.py ──────────────────────────────────────────────────
write_file("src/cnnClassifier/components/evaluation.py", """\
import tensorflow as tf
import mlflow
import mlflow.keras
from urllib.parse import urlparse
from pathlib import Path
from cnnClassifier.entity.config_entity import EvaluationConfig
from cnnClassifier.utils.common import save_json

class Evaluation:
    \'\'\'Evaluates the trained model and logs metrics + model to MLflow.\'\'\'

    def __init__(self, config: EvaluationConfig):
        self.config = config

    def _valid_generator(self):
        \'\'\'Create a 30 % validation split generator (no augmentation).\'\'\'
        datagenerator_kwargs = dict(rescale=1.0 / 255, validation_split=0.30)
        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )
        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )
        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)

    def evaluation(self):
        \'\'\'Load trained model, run evaluation, store loss & accuracy.\'\'\'
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = self.model.evaluate(self.valid_generator)

    def save_score(self):
        \'\'\'Persist evaluation scores to scores.json in the project root.\'\'\'
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)

    def log_into_mlflow(self):
        \'\'\'Log hyperparams, metrics, and the Keras model to MLflow / DagsHub.\'\'\'
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics({
                "loss": self.score[0],
                "accuracy": self.score[1]
            })
            # DagsHub (non-file store) supports the Model Registry
            if tracking_url_type != "file":
                mlflow.keras.log_model(
                    self.model, "model",
                    registered_model_name="VGG16KidneyTumorModel"
                )
            else:
                mlflow.keras.log_model(self.model, "model")
""")

# ── setup.py ──────────────────────────────────────────────────────────────────
write_file("setup.py", """\
import setuptools

setuptools.setup(
    name="cnnClassifier",
    version="0.0.0",
    author="KidneyTumorSystem",
    description="CNN-based Kidney Tumor Classification",
    package_dir={\"\": \"src\"},
    packages=setuptools.find_packages(where=\"src\")
)
""")

print("\n✅ All source files written successfully.")


Writing package source files...
  ✅ src/cnnClassifier/__init__.py
  ✅ src/cnnClassifier/components/__init__.py
  ✅ src/cnnClassifier/config/__init__.py
  ✅ src/cnnClassifier/constants/__init__.py
  ✅ src/cnnClassifier/entity/__init__.py
  ✅ src/cnnClassifier/pipeline/__init__.py
  ✅ src/cnnClassifier/utils/__init__.py
  ✅ src/cnnClassifier/constants/__init__.py
  ✅ src/cnnClassifier/utils/common.py
  ✅ src/cnnClassifier/entity/config_entity.py
  ✅ src/cnnClassifier/config/configuration.py
  ✅ src/cnnClassifier/components/data_ingestion.py
  ✅ src/cnnClassifier/components/prepare_base_model.py
  ✅ src/cnnClassifier/components/prepare_callbacks.py
  ✅ src/cnnClassifier/components/training.py
  ✅ src/cnnClassifier/components/evaluation.py
  ✅ setup.py

✅ All source files written successfully.


In [6]:
# : Install the cnnClassifier package from local source ───────────────
# pip install -e installs in editable/development mode so any in-notebook
# edits to src/ are immediately reflected without re-installing.

import subprocess, sys, os
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".", "-q"],
    capture_output=True, text=True
)
print(result.stdout or "✅ cnnClassifier installed (editable mode)")
if result.returncode != 0:
    print("STDERR:", result.stderr)


✅ cnnClassifier installed (editable mode)


In [8]:
# ──  Stage 1 — Data Ingestion ─────────────────────────────────────────
# Downloads the kidney CT-scan dataset (~57 MB) from Google Drive via gdown.
# Extracted structure: artifacts/data_ingestion/kidney-ct-scan-image/{Tumor,Normal}/
# Skips download if the zip already exists on disk (idempotent).

import os
from pathlib import Path
import sys # Import sys

# Add the project root's 'src' directory to Python path to enable package imports
PROJECT_ROOT = "/content/KidneyTumorSystem" # Re-define PROJECT_ROOT for robustness in this cell
project_root_src = os.path.join(PROJECT_ROOT, "src")
if project_root_src not in sys.path:
    sys.path.insert(0, project_root_src)

from cnnClassifier import logger
from cnnClassifier.config.configuration import ConfigurationManager
from cnnClassifier.components.data_ingestion import DataIngestion

logger.info(">>>>>> Stage 1: Data Ingestion started <<<<<<")

try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)

    # Only download if the zip is not already present (saves time on re-runs)
    if not Path(data_ingestion_config.local_data_file).exists():
        data_ingestion.download_file()
    else:
        logger.info("Zip already present — skipping download.")

    data_ingestion.extract_zip_file()

except Exception as e:
    logger.exception(e)
    raise e

logger.info(">>>>>> Stage 1: Data Ingestion completed <<<<<<\n")

# ── Quick sanity check ────────────────────────────────────────────────────────
import os
base = "artifacts/data_ingestion/kidney-ct-scan-image"
for cls in os.listdir(base):
    n = len(os.listdir(os.path.join(base, cls)))
    print(f"  Class '{cls}': {n} images")


Downloading...
From (original): https://drive.google.com/uc?/export=download&id=1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3
From (redirected): https://drive.google.com/uc?%2Fexport=download&id=1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3&confirm=t&uuid=f20edf24-96cf-4757-831c-640c288c14ad
To: /content/KidneyTumorSystem/artifacts/data_ingestion/data.zip
100%|██████████| 57.7M/57.7M [00:01<00:00, 38.9MB/s]


  Class 'Tumor': 225 images
  Class 'Normal': 240 images


In [9]:
# ── : Stage 2 — Prepare Base Model ─────────────────────────────────────
# Loads VGG16 with ImageNet weights (downloads ~550 MB on first run).
# Freezes all convolutional layers and attaches a 2-class softmax head.
# Saves: base_model.h5 (no head) and base_model_updated.h5 (with head).

from cnnClassifier import logger
from cnnClassifier.config.configuration import ConfigurationManager
from cnnClassifier.components.prepare_base_model import PrepareBaseModel

logger.info(">>>>>> Stage 2: Prepare Base Model started <<<<<<")

try:
    config = ConfigurationManager()
    prepare_base_model_config = config.get_prepare_base_model_config()
    prepare_base_model = PrepareBaseModel(config=prepare_base_model_config)
    prepare_base_model.get_base_model()       # Download VGG16
    prepare_base_model.update_base_model()    # Attach classification head

except Exception as e:
    logger.exception(e)
    raise e

logger.info(">>>>>> Stage 2: Prepare Base Model completed <<<<<<\n")


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │        50,178 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,764,866 (56.32 MB)

 Trainable params: 50,178 (196.01 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [10]:
# ── : Stage 3 — Prepare Callbacks ──────────────────────────────────────
# Creates two callbacks:
#   1. TensorBoard  — writes event logs to a timestamped directory
#   2. ModelCheckpoint — saves best model weights (by val_loss) to .h5
# The callback objects are kept in `callback_list` for the training stage.

from cnnClassifier import logger
from cnnClassifier.config.configuration import ConfigurationManager
from cnnClassifier.components.prepare_callbacks import PrepareCallback

logger.info(">>>>>> Stage 3: Prepare Callbacks started <<<<<<")

try:
    config = ConfigurationManager()
    prepare_callbacks_config = config.get_prepare_callback_config()
    prepare_callbacks = PrepareCallback(config=prepare_callbacks_config)
    callback_list = prepare_callbacks.get_tb_ckpt_callbacks()
    print(f"✅ Callbacks ready: {[type(cb).__name__ for cb in callback_list]}")

except Exception as e:
    logger.exception(e)
    raise e

logger.info(">>>>>> Stage 3: Prepare Callbacks completed <<<<<<\n")


✅ Callbacks ready: ['TensorBoard', 'ModelCheckpoint']


In [12]:
# ── : Stage 4 — Model Training ────────────────────────────────────────
# Trains the VGG16 model using the kidney CT-scan images.
# GPU is used automatically when available (set runtime to T4 GPU).
# Trained model saved to: artifacts/training/model.h5
# Best checkpoint saved to: artifacts/prepare_callbacks/checkpoint_dir/model.h5
#
# Hyperparameters are read from params.yaml — adjust EPOCHS there for longer runs.

import tensorflow as tf
from cnnClassifier import logger
from cnnClassifier.config.configuration import ConfigurationManager
from cnnClassifier.components.training import Training

# Confirm GPU is being used
gpus = tf.config.list_physical_devices("GPU")
print(f"🖥️  Training on: {'GPU — ' + gpus[0].name if gpus else 'CPU (no GPU detected)'}")

logger.info(">>>>>> Stage 4: Training started <<<<<<")

try:
    config_manager = ConfigurationManager() # Renamed to avoid confusion with internal 'config'
    training_config = config_manager.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()           # Load updated model from disk

    # --- FIX START ---
    # Recompile the model to correctly link the optimizer with its variables
    # The learning rate is needed for SGD optimizer.
    learning_rate = config_manager.params.LEARNING_RATE
    training.model.compile(
        optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
        loss=tf.keras.losses.CategoricalCrossentropy(),
        metrics=["accuracy"]
    )
    logger.info(f"Model recompiled successfully with learning rate: {learning_rate}")
    # --- FIX END ---

    training.train_valid_generator()    # Build train/val data pipelines
    training.train(callback_list=callback_list)  # Fit the model

except Exception as e:
    logger.exception(e)
    raise e

logger.info(">>>>>> Stage 4: Training completed <<<<<<\n")


🖥️  Training on: GPU — /physical_device:GPU:0
Found 93 images belonging to 2 classes.
Found 372 images belonging to 2 classes.
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 483ms/step - accuracy: 0.5606 - loss: 10.1530

23/23 ━━━━━━━━━━━━━━━━━━━━ 23s 577ms/step - accuracy: 0.5955 - loss: 10.2315 - val_accuracy: 0.4000 - val_loss: 18.7183


In [13]:
# ── : Stage 5 — Model Evaluation + MLflow / DagsHub Logging ───────────
# Evaluates the trained model on a 30 % validation split.
# Saves scores to scores.json (loss + accuracy).
# Logs params, metrics, and the Keras model artifact to MLflow / DagsHub.
# The MLflow URI is picked up from the env variable set in Cell 3.

from cnnClassifier import logger
from cnnClassifier.config.configuration import ConfigurationManager
from cnnClassifier.components.evaluation import Evaluation

logger.info(">>>>>> Stage 5: Model Evaluation started <<<<<<")

try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()     # Runs model.evaluate() and stores score
    evaluation.save_score()     # Writes scores.json
    evaluation.log_into_mlflow()  # Pushes to DagsHub MLflow

except Exception as e:
    logger.exception(e)
    raise e

logger.info(">>>>>> Stage 5: Model Evaluation completed <<<<<<\n")

# ── Print results ─────────────────────────────────────────────────────────────
import json
with open("scores.json") as f:
    scores = json.load(f)
print(f"\n📊 Evaluation Results:")
print(f"   Loss     : {scores['loss']:.4f}")
print(f"   Accuracy : {scores['accuracy']:.4f}")


Found 139 images belonging to 2 classes.
9/9 ━━━━━━━━━━━━━━━━━━━━ 8s 782ms/step - accuracy: 0.4820 - loss: 15.8810


2026/04/10 04:14:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/10 04:14:19 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.
Successfully registered model 'VGG16KidneyTumorModel'.
2026/04/10 04:14:41 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: VGG16KidneyTumorModel, version 1
Created version '1' of model 'VGG16KidneyTumorModel'.


🏃 View run melodic-pig-23 at: https://dagshub.com/prithusarkar90/networksecurity.mlflow/#/experiments/0/runs/cee5e5a46c8147968b4b35f7846c00a5
🧪 View experiment at: https://dagshub.com/prithusarkar90/networksecurity.mlflow/#/experiments/0

📊 Evaluation Results:
   Loss     : 15.8810
   Accuracy : 0.4820


In [14]:
# ──: Push experiment metadata to MongoDB ─────────────────────────────
# Records a summary document (run ID, scores, params, timestamps) to MongoDB
# so that experiment history is persisted outside the ephemeral Colab VM.
# Collection: kidney_tumor_db.experiments

import os, json, datetime
from pymongo import MongoClient
from pymongo.errors import ConnectionFailure

MONGO_URL = os.environ.get("MONGO_DB_URL", "")

try:
    client = MongoClient(MONGO_URL, serverSelectionTimeoutMS=5000)
    client.admin.command("ping")   # Quick connectivity check
    db = client["kidney_tumor_db"]
    collection = db["experiments"]

    # Load scores written by the evaluation stage
    with open("scores.json") as f:
        scores = json.load(f)

    # Build the metadata document
    doc = {
        "timestamp"     : datetime.datetime.utcnow().isoformat(),
        "mlflow_uri"    : os.environ.get("MLFLOW_TRACKING_URI"),
        "scores"        : scores,
        "params"        : {
            "epochs"        : 1,
            "batch_size"    : 16,
            "image_size"    : [224, 224, 3],
            "learning_rate" : 0.01,
            "augmentation"  : True,
            "classes"       : 2,
            "weights"       : "imagenet"
        },
        "artifacts": {
            "trained_model"    : "artifacts/training/model.h5",
            "base_model"       : "artifacts/prepare_base_model/base_model.h5",
            "updated_model"    : "artifacts/prepare_base_model/base_model_updated.h5",
            "checkpoint_model" : "artifacts/prepare_callbacks/checkpoint_dir/model.h5",
        }
    }

    result = collection.insert_one(doc)
    print(f"✅ Metadata inserted to MongoDB (id: {result.inserted_id})")

except ConnectionFailure as e:
    # Non-fatal: print warning but continue so the rest of the notebook runs
    print(f"⚠️  MongoDB connection failed — skipping metadata push.\n   Error: {e}")
except Exception as e:
    print(f"⚠️  Unexpected MongoDB error: {e}")


/tmp/ipykernel_3573/824572847.py:24: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp"     : datetime.datetime.utcnow().isoformat(),


✅ Metadata inserted to MongoDB (id: 69d8793af348064559400d21)


In [15]:
# ── : Write metadata.txt snapshot ─────────────────────────────────────
# Saves a human-readable text snapshot of all run details.
# This file is included in the final zip so it can be reviewed on GitHub
# without loading the model — avoids large-binary rendering issues in git.

import os, json, datetime

with open("scores.json") as f:
    scores = json.load(f)

metadata_content = f"""
========================================
  Kidney Tumor Identification System
  Experiment Metadata Snapshot
========================================

Run Timestamp   : {datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')} UTC
MLflow URI      : {os.environ.get('MLFLOW_TRACKING_URI', 'N/A')}

--- Evaluation Scores ---
Loss            : {scores['loss']:.6f}
Accuracy        : {scores['accuracy']:.6f}

--- Hyperparameters ---
EPOCHS          : 1
BATCH_SIZE      : 16
IMAGE_SIZE      : [224, 224, 3]
LEARNING_RATE   : 0.01
AUGMENTATION    : True
CLASSES         : 2
WEIGHTS         : imagenet
INCLUDE_TOP     : False

--- Artifact Paths ---
Base Model      : artifacts/prepare_base_model/base_model.h5
Updated Model   : artifacts/prepare_base_model/base_model_updated.h5
Checkpoint      : artifacts/prepare_callbacks/checkpoint_dir/model.h5
Trained Model   : artifacts/training/model.h5
Scores File     : scores.json
Logs            : logs/running_logs.log
TensorBoard     : artifacts/prepare_callbacks/tensorboard_log_dir/

========================================
"""

with open("metadata.txt", "w") as f:
    f.write(metadata_content)

print(metadata_content)



  Kidney Tumor Identification System
  Experiment Metadata Snapshot

Run Timestamp   : 2026-04-10 04:14:56 UTC
MLflow URI      : https://dagshub.com/prithusarkar90/networksecurity.mlflow

--- Evaluation Scores ---
Loss            : 15.881035
Accuracy        : 0.482014

--- Hyperparameters ---
EPOCHS          : 1
BATCH_SIZE      : 16
IMAGE_SIZE      : [224, 224, 3]
LEARNING_RATE   : 0.01
AUGMENTATION    : True
CLASSES         : 2
WEIGHTS         : imagenet
INCLUDE_TOP     : False

--- Artifact Paths ---
Base Model      : artifacts/prepare_base_model/base_model.h5
Updated Model   : artifacts/prepare_base_model/base_model_updated.h5
Checkpoint      : artifacts/prepare_callbacks/checkpoint_dir/model.h5
Trained Model   : artifacts/training/model.h5
Scores File     : scores.json
Logs            : logs/running_logs.log
TensorBoard     : artifacts/prepare_callbacks/tensorboard_log_dir/




/tmp/ipykernel_3573/271181702.py:17: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  Run Timestamp   : {datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')} UTC


In [16]:
# ── : Zip all outputs and trigger browser download ─────────────────────
# Collects every important artifact, config, score, and log into a single zip.
# Large model .h5 files are included so the run is fully reproducible.
# The notebook itself is also added so the zip is self-contained.
#
# Files included:
#   ├── artifacts/           (models, checkpoints, tensorboard logs, data)
#   ├── config/config.yaml
#   ├── params.yaml
#   ├── scores.json
#   ├── metadata.txt
#   ├── logs/running_logs.log
#   └── src/                 (full package source for reference)

import os, zipfile, datetime
from google.colab import files

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
zip_name = f"/content/KidneyTumor_outputs_{timestamp}.zip"

# Directories and files to include
include_paths = [
    "artifacts",
    "config",
    "src",
    "logs",
    "scores.json",
    "metadata.txt",
    "params.yaml",
    "setup.py",
]

print(f"📦 Creating zip: {zip_name}")

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for item in include_paths:
        if os.path.isfile(item):
            zf.write(item)
            print(f"  + {item}")
        elif os.path.isdir(item):
            for root, dirs, fnames in os.walk(item):
                # Skip __pycache__ and .git directories
                dirs[:] = [d for d in dirs if d not in {"__pycache__", ".git"}]
                for fname in fnames:
                    fpath = os.path.join(root, fname)
                    zf.write(fpath)
            print(f"  + {item}/  (directory)")

zip_size_mb = os.path.getsize(zip_name) / (1024 ** 2)
print(f"\n✅ Zip created: {zip_name}  ({zip_size_mb:.1f} MB)")
print("⬇️  Triggering download...")

# Trigger automatic browser download
files.download(zip_name)
print("\n🎉 Pipeline complete! All outputs downloaded.")


📦 Creating zip: /content/KidneyTumor_outputs_20260410_041505.zip
  + artifacts/  (directory)
  + config/  (directory)
  + src/  (directory)
  + logs/  (directory)
  + scores.json
  + metadata.txt
  + params.yaml
  + setup.py

✅ Zip created: /content/KidneyTumor_outputs_20260410_041505.zip  (319.3 MB)
⬇️  Triggering download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 Pipeline complete! All outputs downloaded.
